In [ ]:
"""
E-Commerce Intelligence Suite
==============================
01_data_generation.py

Was dieses Script macht:
    Laedt den UCI Online Retail Dataset (data_300k.csv) und erweitert ihn
    um 9 synthetische Spalten die ein realistisches E-Commerce Marketing-
    Umfeld simulieren: Email-Events, UTM-Sources, A/B-Test-Flags und Churn-Labels.

Business Impact:
    In echten Projekten fehlen oft Marketing-Dimensionen in Rohdaten.
    Dieses Script zeigt wie man fehlende Daten realistisch modelliert --
    relevant fuer Data Engineering, Pipeline-Design und Dashboard-Vorbereitung.

Voraussetzungen:
    Python 3.9+
    pip install pandas numpy

Output:
    data/data_enriched_fixed.csv  (300k Zeilen, 17 Spalten)

Ausfuehren:
    python python/01_data_generation.py
"""

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------------ #
# Konfiguration
# ------------------------------------------------------------------ #
RANDOM_SEED   = 42
INPUT_FILE    = 'data/data_300k.csv'
OUTPUT_FILE   = 'data/data_enriched_fixed.csv'
CHURN_DAYS    = 60   # Kunden die laenger als X Tage inaktiv sind = churned

np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------------ #
# 1. Daten laden
# ------------------------------------------------------------------ #
print('Lade Rohdaten...')
df = pd.read_csv(INPUT_FILE, encoding='cp1252')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'  {len(df):,} Zeilen geladen')


# ------------------------------------------------------------------ #
# 2. Campaign Name (zeitbasiert)
# ------------------------------------------------------------------ #
# Business Impact: Kampagnenzuordnung ist in echten Systemen oft eine
# Lookup-Tabelle. Hier simulieren wir sie zeitbasiert -- realistisch
# fuer saisonale E-Commerce Shops.
def assign_campaign(date):
    m, d = date.month, date.day
    if m == 12 and d <= 10: return 'BlackFriday_Dec2010'
    if m == 12 and d <= 24: return 'Xmas2010'
    if m == 12:             return 'NYE2010'
    if m == 1:              return 'NewYear2011'
    if m == 2:              return 'Valentines2011'
    if m == 3:              return 'Spring2011'
    if m == 4:              return 'Easter2011'
    if m == 5:              return 'MayBank2011'
    if m == 6:              return 'Summer2011'
    if m == 7:              return 'SummerSale2011'
    return 'August2011'

df['campaign_name'] = df['InvoiceDate'].apply(assign_campaign)
print(f'  Kampagnen zugewiesen: {df["campaign_name"].nunique()} unique')


# ------------------------------------------------------------------ #
# 3. UTM Source & Medium
# ------------------------------------------------------------------ #
# Business Impact: UTM-Parameter sind die Basis fuer Attribution-Modelle.
# Ohne sie weiss ein Kunde nicht welcher Kanal seinen Umsatz treibt.
UTM_SOURCES  = ['email', 'google', 'direct', 'instagram', 'referral']
UTM_WEIGHTS  = [0.35,    0.30,     0.20,     0.10,        0.05]
UTM_MEDIUM   = {
    'email':     'newsletter',
    'google':    'cpc',
    'direct':    'none',
    'instagram': 'social',
    'referral':  'referral'
}

df['utm_source'] = np.random.choice(UTM_SOURCES, size=len(df), p=UTM_WEIGHTS)
df['utm_medium'] = df['utm_source'].map(UTM_MEDIUM)
print(f'  UTM Sources: {df["utm_source"].value_counts().to_dict()}')


# ------------------------------------------------------------------ #
# 4. A/B Test Flag
# ------------------------------------------------------------------ #
# Business Impact: A/B-Test-Flags ermoeglichen statistische Auswertung
# von Feature-Experimenten (z.B. neuer Checkout). Deterministisch per
# CustomerID -- jeder Kunde sieht immer dieselbe Variante.
df['ab_variant'] = np.where(
    df['CustomerID'].notna(),
    np.where(df['CustomerID'] % 2 == 0, 'A', 'B'),
    'A'
)
print(f'  A/B Split: {df["ab_variant"].value_counts().to_dict()}')


# ------------------------------------------------------------------ #
# 5. Email Events (Sent, Opened, Clicked)
# ------------------------------------------------------------------ #
# Business Impact: Email-Funnel-Daten sind die Grundlage fuer
# Marketing-ROI-Berechnungen. Open Rate ~62%, CTOR ~45% entsprechen
# realistischen Branchenwerten fuer E-Commerce.
is_email = df['utm_source'] == 'email'

df['email_sent'] = is_email.astype(int)

# Opened: nur wenn email_sent = 1
df['email_opened'] = 0
df.loc[is_email, 'email_opened'] = np.random.choice(
    [0, 1], size=is_email.sum(), p=[0.38, 0.62]
)

# Clicked: nur wenn email_opened = 1
opened = df['email_opened'] == 1
df['email_clicked'] = 0
df.loc[opened, 'email_clicked'] = np.random.choice(
    [0, 1], size=opened.sum(), p=[0.55, 0.45]
)

open_rate = df['email_opened'].sum() / df['email_sent'].sum() * 100
ctor      = df['email_clicked'].sum() / df['email_opened'].sum() * 100
print(f'  Email Funnel: Open Rate {open_rate:.1f}% · CTOR {ctor:.1f}%')


# ------------------------------------------------------------------ #
# 6. Churn Label
# ------------------------------------------------------------------ #
# Business Impact: Churn-Labels sind die Zielvariable fuer
# Prediction-Modelle. Definition: Kein Kauf in den letzten X Tagen
# relativ zum letzten Datum im Datensatz.
max_date = df['InvoiceDate'].max()

customer_last = (
    df.groupby('CustomerID')['InvoiceDate']
    .max()
    .reset_index()
    .rename(columns={'InvoiceDate': 'last_purchase'})
)
customer_last['days_since'] = (max_date - customer_last['last_purchase']).dt.days
customer_last['churned']    = (customer_last['days_since'] > CHURN_DAYS).astype(int)

df = df.merge(customer_last[['CustomerID', 'churned']], on='CustomerID', how='left')
df['churned'] = df['churned'].fillna(1).astype(int)

churn_rate = customer_last['churned'].mean() * 100
print(f'  Churn Rate: {churn_rate:.1f}%')


# ------------------------------------------------------------------ #
# 7. Revenue Spalte + Customer ID bereinigen
# ------------------------------------------------------------------ #
df['revenue']     = (df['Quantity'] * df['UnitPrice']).round(2)
df['customer_id'] = df['CustomerID'].fillna(0).astype(int)
df = df.drop(columns=['CustomerID'])

# Spalten lowercase
df.columns = [c.lower().replace(' ', '_') for c in df.columns]


# ------------------------------------------------------------------ #
# 8. Output speichern
# ------------------------------------------------------------------ #
os.makedirs('data', exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)
print(f'\nFertig! Gespeichert: {OUTPUT_FILE}')
print(f'Shape: {df.shape}')
print(f'Spalten: {df.columns.tolist()}')

: 